<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Hierarchical Clustering</h1></center>

Hierarchical clustering builds a **tree of clusters** (dendrogram) without requiring a pre-specified k.

### Topics:
1. [Theory — Agglomerative Clustering](#1)
2. [Dataset & Visualization](#2)
3. [Dendrogram](#3)
4. [Linkage Methods Compared](#4)
5. [Evaluation of Model](#5)

In [ ]:
import seaborn as sns
hc_colors = ['#240046', '#5a189a', '#9d4edd', '#c77dff', '#e0aaff']
print("Hierarchical Clustering Colors:")
sns.palplot(sns.color_palette(hc_colors))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, cophenet
from scipy.spatial.distance import pdist
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.datasets import load_iris, make_blobs
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
np.random.seed(42)
print("Libraries loaded.")

## Let's Start!

<a id = "1"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Theory — Agglomerative Clustering</h1></center>

## How It Works

Agglomerative clustering starts with **n singleton clusters** and merges the two closest at each step until only one cluster remains.

## Linkage Methods

| Linkage | Distance Between Clusters | Tendency |
|---|---|---|
| Single | Minimum pairwise distance | Long chained clusters |
| Complete | Maximum pairwise distance | Compact, spherical |
| Average | Mean pairwise distance | Compromise |
| Ward | Minimises within-cluster variance | Usually best |

## Cophenetic Correlation

Measures how faithfully the dendrogram preserves original pairwise distances. Higher = better summary.

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'],['\\(','\\)']]}});</SCRIPT>

<a id = "2"></a>
<center><h1 style = "background:#240046 ;color:white;border:0;font-weight:bold">Information About Dataset</h1></center>

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
X_iris = df.drop(columns="target")
y_iris = df["target"]

sc = StandardScaler()
X_scaled = sc.fit_transform(X_iris)

print(f"Shape: {df.shape}")
print(f"Classes: {iris.target_names}")
df.describe()

<center><h1 style = "background:#5a189a ;color:white;border:0;font-weight:bold">Data Visualization</h1></center>

In [ ]:
plt.figure(dpi=100)
sns.pairplot(df, hue="target",
             palette={0:"#9d4edd",1:"#c77dff",2:"#e0aaff"},
             diag_kind="kde")
plt.suptitle("Iris Pairplot", y=1.01)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5), dpi=100)
sns.heatmap(X_iris.corr(numeric_only=True), annot=True, fmt=".2f",
            cmap="Purples", linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.show()

<a id = "3"></a>
<center><h1 style = "background:#9d4edd ;color:white;border:0;font-weight:bold">Dendrogram</h1></center>

In [ ]:
Z_ward = linkage(X_scaled, method="ward")

fig, ax = plt.subplots(figsize=(14, 5), dpi=100)
dendrogram(Z_ward, ax=ax, truncate_mode="level", p=5,
           color_threshold=0.7 * max(Z_ward[:, 2]))
ax.set_title("Dendrogram — Ward Linkage (Iris)", fontweight="bold")
ax.set_xlabel("Sample / Cluster size")
ax.set_ylabel("Distance")
ax.axhline(y=3.5, color="#ff006e", ls="--", label="Cut → 3 clusters")
ax.legend()
plt.tight_layout(); plt.show()

<a id = "4"></a>
<center><h1 style = "background:#c77dff ;color:black;border:0;font-weight:bold">Linkage Methods Compared</h1></center>

In [ ]:
linkages = ["ward", "complete", "average", "single"]
palette = ["#240046", "#5a189a", "#9d4edd", "#e0aaff"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), dpi=100)

for ax, link, color in zip(axes, linkages, palette):
    model = AgglomerativeClustering(n_clusters=3, linkage=link)
    labels = model.fit_predict(X_scaled)
    ari = adjusted_rand_score(y_iris, labels)
    sil = silhouette_score(X_scaled, labels)
    ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels,
               cmap="viridis", s=25, alpha=0.8)
    ax.set_title(f"{link.capitalize()}\nARI={ari:.3f}  Sil={sil:.3f}",
                 fontweight="bold")

plt.suptitle("Hierarchical Clustering — 4 Linkage Methods (k=3)", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Cophenetic correlation per linkage
pdist_iris = pdist(X_scaled, metric="euclidean")
coph_results = []
for link in linkages:
    Z = linkage(X_scaled, method=link)
    c, _ = cophenet(Z, pdist_iris)
    coph_results.append({"Linkage": link, "Cophenetic Correlation": round(c, 4)})
pd.DataFrame(coph_results)

<a id = "5"></a>
<center><h1 style = "background:#e0aaff ;color:black;border:0;font-weight:bold">Evaluation of Model</h1></center>

In [ ]:
best_model = AgglomerativeClustering(n_clusters=3, linkage="ward")
labels = best_model.fit_predict(X_scaled)

ari = adjusted_rand_score(y_iris, labels)
sil = silhouette_score(X_scaled, labels)
print(f"Adjusted Rand Index : {ari:.4f}")
print(f"Silhouette Score    : {sil:.4f}")

pd.DataFrame({"Metric": ["Silhouette Score", "Adjusted Rand Index"],
              "Score":  [sil, ari]}).round(4)

In [ ]:
cross = pd.crosstab(
    y_iris.map({0:"setosa",1:"versicolor",2:"virginica"}),
    pd.Series(labels).map({i: f"Cluster {i}" for i in range(3)}),
    rownames=["True Species"], colnames=["Cluster"]
)
print(cross)